# EDA — GRAZPEDWRI-DX

Exploração inicial do dataset preparado para YOLO (`dataset/yolo/`).

**Kernel:** TCC YOLO (venv)

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import yaml

CODIGO = Path("..").resolve()
ARCHIVE = CODIGO.parent / "archive"
YOLO_DIR = CODIGO / "dataset" / "yolo"
DATA_YAML = YOLO_DIR / "data.yaml"

with open(DATA_YAML, encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

print("Classes:", cfg["names"])
for split in ("train", "val", "test"):
    n = len(list((YOLO_DIR / "images" / split).glob("*.png")))
    print(f"{split}: {n} imagens")

In [ ]:
df = pd.read_csv(ARCHIVE / "dataset.csv")
print(f"Pacientes: {df['patient_id'].nunique()}")
print(f"Imagens: {len(df)}")
print(df["gender"].value_counts())
print("Fratura visível (CSV):", df["fracture_visible"].notna().sum())

In [ ]:
from collections import Counter

class_names = cfg["names"]
counts = Counter()
label_dir = YOLO_DIR / "labels" / "train"
for lbl in label_dir.glob("*.txt"):
    text = lbl.read_text(encoding="utf-8").strip()
    if not text:
        continue
    for line in text.splitlines():
        cls = int(line.split()[0])
        counts[class_names[cls]] += 1

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(list(counts.keys()), list(counts.values()), color="#2c6e9d")
ax.set_title("Anotações por classe (split train)")
ax.tick_params(axis="x", rotation=35)
plt.tight_layout()
plt.show()

In [ ]:
import cv2
import random

img_dir = YOLO_DIR / "images" / "train"
lbl_dir = YOLO_DIR / "labels" / "train"
samples = random.sample(list(img_dir.glob("*.png")), k=3)

for img_path in samples:
    img = cv2.imread(str(img_path), cv2.IMREAD_UNCHANGED)
    if img is None:
        continue
    if img.dtype == "uint16":
        vis = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX).astype("uint8")
    else:
        vis = img.copy()
    h, w = vis.shape[:2]
    lbl_path = lbl_dir / f"{img_path.stem}.txt"
    if lbl_path.exists():
        for line in lbl_path.read_text(encoding="utf-8").splitlines():
            parts = line.split()
            if len(parts) < 5:
                continue
            cls, xc, yc, bw, bh = parts[:5]
            x1 = int((float(xc) - float(bw) / 2) * w)
            y1 = int((float(yc) - float(bh) / 2) * h)
            x2 = int((float(xc) + float(bw) / 2) * w)
            y2 = int((float(yc) + float(bh) / 2) * h)
            cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 255, 0), 2)
    plt.figure(figsize=(5, 5))
    plt.imshow(vis, cmap="gray")
    plt.title(img_path.stem)
    plt.axis("off")
    plt.show()